In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

# Test script
this is the test script of the regional location and bias correction
*At GCM resolution prediction, no bias correction invloved.
*downscale resolution, there is bias correction, which is on progress.

In [ ]:
# read values from low ice model
mean_values_low = []
senario = 'SSP585' #["natural","SSP126","SSP245","SSP370","SSP460","SSP534","SSP585"]
variable = "tas"  # "tas","precip","evap","ice_sheet","iceconc","LAI_PFT","sm","snowdepth","soiltemp","windspeed"
site = 'UK' # or 'UK' 
if site == 'UK':
    site_lon_index = -1
    site_lat_index = 14
    # site_lon_indexs = [-2,-1,-1,-1,0]
    # site_lat_indexs = [15,15,14,13,15]
    site_lon_index_UK_sea = [-1,-2,-3,-2,-2,-1, 0, 0, 1]
    site_lat_index_UK_sea = [16,16,15,14,13,12,13,14,15]


bias_correction = False # no bias correction before downscaling
member = 67
# Load the NetCDF data for each file
path="/Users/bo20541/Library/CloudStorage/OneDrive-UniversityofBristol/TONIC-Oligocene/Downscaling/training_data_highres/"
data = xr.open_dataset(path+f'emul_in_Y_modhighice_{variable}.nc')

# Read the variable "mean" from the dataset
mean_variable = data['var']
latitude = data['lat'].values
longitude = data['lon'].values
# Append the mean values to the list
mean_values_low.append(mean_variable.astype("float64").values)

# Convert the list of mean values into a NumPy array for the ensemble
ensemble_array_low = np.array(mean_values_low, dtype=np.float64)
# Print the shape of the ensemble array to verify
print("Ensemble array shape:", ensemble_array_low.shape)


In [ ]:
# read the mean values from the NetCDF files for the high ice concentration ensemble
mean_values_high = []

# Load the NetCDF data for each file
data = xr.open_dataset(path+f'emul_in_Y_modhighice_{variable}.nc')

# Read the variable "mean" from the dataset
mean_variable = data['var'].astype("float64")  # Ensure the data type is float64 as Required
# Append the mean values to the listti a a 
mean_values_high.append(mean_variable.astype("float64").values)

# Convert the list of mean values into a NumPy array for the ensemble
ensemble_array_high = np.array(mean_values_high, dtype=np.float64)
# Print the shape of the ensemble array to verify
print("Ensemble array shape:", ensemble_array_high.shape)

In [ ]:

# ensemble low and high results
ice_all = []

prediction_path = f"/Users/bo20541/Library/CloudStorage/OneDrive-UniversityofBristol/TONIC-Oligocene/Emulator_Xin/2025_Bristol_5D_v002_Xin/Conceptual-GSL/PosivaSKB-master/Results"
prediction_input = f"{prediction_path}/emul_inputs_{senario}.{member}.updated.res"
emu_input_data = pd.read_csv(prediction_input, sep='\s+', header=0)
ice = emu_input_data.iloc[:, 4].values
ice_all.append(ice)

ice_all = np.array(ice_all)  # Convert the list to a NumPy array
print(ice_all.shape)
print("the ice data looks like this:",ice_all[:])

# select data according to the ice concentration
# Initialize an empty list to collect the data
ensemble_array_all = []

for y in range(ice.shape[0]):  # Adjusted range to match Python's zero-based indexing
    if ice_all[0][y] < 0.0:
        ensemble_array_all.append(ensemble_array_high[0, y, :, :])
    else:
        ensemble_array_all.append(ensemble_array_low[0, y, :, :])

ensemble_array_all = np.array(ensemble_array_all, dtype=np.float64)  # Convert the list to a NumPy array after appending

# Convert the list to a NumPy array
ensemble_array = np.array(ensemble_array_all)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([longitude.min(), longitude.max(), 
               latitude.min(), latitude.max()], crs=ccrs.PlateCarree())

# Add coastlines and land features
ax.coastlines(resolution='50m', color='black', linewidth=0.8)
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.2)

# Plot data
mesh = ax.pcolormesh(longitude, latitude, ensemble_array[700, :, :], 
                     shading='auto', cmap='coolwarm', transform=ccrs.PlateCarree())
plt.colorbar(mesh, ax=ax, label='Value', shrink=0.8)

# Add scatter point
ax.scatter(longitude[site_lon_index + (longitude.shape[0] // 2)], 
           latitude[site_lat_index], color='red', s=100, marker='o', 
           edgecolors='black', linewidths=1.5, label='Point [21,34]', 
           transform=ccrs.PlateCarree(), zorder=5)

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Map Plot of ensemble_array[700,:,:] with Coastlines')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

lat_min, lat_max = latitude[site_lat_index] - 20.0, latitude[site_lat_index] + 20.0
lon_min, lon_max = longitude[site_lon_index + (longitude.shape[0] // 2)-1] - 20.0, longitude[site_lon_index + (longitude.shape[0] // 2)-1] + 20.0

# Create a mask for the latitude and longitude values
lat_mask = (latitude >= lat_min) & (latitude <= lat_max)
lon_mask = (longitude >= lon_min) & (longitude <= lon_max)

regional_pi_lat = latitude[lat_mask]
regional_pi_lon = longitude[lon_mask]
regional_ensemble_array = ensemble_array[:, lat_mask, :][:, :, lon_mask]

# Create cartopy map with coastlines
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

# Add coastlines and geographic features
ax.coastlines(resolution='50m', color='black', linewidth=0.8)
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.2)

# Plot data
mesh = ax.pcolormesh(regional_pi_lon, regional_pi_lat, regional_ensemble_array[700, :, :], 
                     shading='auto', cmap='coolwarm', transform=ccrs.PlateCarree())
plt.colorbar(mesh, ax=ax, label='Value', shrink=0.8)

# Add site marker
ax.scatter(longitude[site_lon_index + (longitude.shape[0] // 2)-1], 
           latitude[site_lat_index], color='red', s=100, marker='o',
           edgecolors='black', linewidths=1.5, 
           label=f'Point {site_lat_index},{site_lon_index}', 
           transform=ccrs.PlateCarree(), zorder=5)

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Regional Map Plot with Coastlines')
plt.legend()
plt.tight_layout()
plt.show()

# Multi member cases 

In [ ]:
# plot the value 
import matplotlib.pyplot as plt
plt.figure(figsize=(20, 5))
plt.plot(var_kaeri_ano[:], color='red', label='best member')
plt.xlabel('Time AP')
plt.ylabel(f'{variable} anomaly')
plt.legend()
plt.title(f'{site} {variable} anomaly {senario}')
plt.grid()
plt.xlim(1,1001)
plt.ylim(-10, 10)
plt.show()
